In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=4000, n_features=20, n_informative=8, n_redundant=4,
    class_sep=1.3, flip_y=0.02, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(X_train.shape, X_test.shape)


(3000, 20) (1000, 20)


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

clf = LogisticRegression(max_iter=200, solver='lbfgs', n_jobs=-1)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:,1]
print(classification_report(y_test, pred))
print('ROC AUC:', roc_auc_score(y_test, proba))

              precision    recall  f1-score   support

           0       0.80      0.83      0.81       494
           1       0.83      0.79      0.81       506

    accuracy                           0.81      1000
   macro avg       0.81      0.81      0.81      1000
weighted avg       0.81      0.81      0.81      1000

ROC AUC: 0.8862716231137284


In [3]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from scipy.stats import loguniform

param_distrib = {
    'C': loguniform(1e-3, 1e1),
    'penalty': ['l2'],
    'solver': ['lbfgs']
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    LogisticRegression(max_iter=400, n_jobs=-1),
    param_distributions=param_distrib, n_iter=20, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)
best = search.best_estimator_
print('Mejor ROC AUC (CV):', search.best_score_)
print('Mejores params:', search.best_params_)


Mejor ROC AUC (CV): 0.8825992266322948
Mejores params: {'C': np.float64(0.03148911647956861), 'penalty': 'l2', 'solver': 'lbfgs'}


In [4]:
from scipy.spatial.distance import euclidean
import numpy as np

def hellinger(p, q):
    # p y q son histogramas normalizados
    return (1/np.sqrt(2)) * np.linalg.norm(np.sqrt(p) - np.sqrt(q))

def feature_drift_score(x_ref, x_new, bins=30):
    # hist normalizados
    p, _ = np.histogram(x_ref, bins=bins, density=True)
    q, _ = np.histogram(x_new, bins=bins, density=True)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return hellinger(p, q)

# Creamos "drift" desplazando media de primeras 3 features
rng = np.random.default_rng(7)
X_prod = X_test.copy()
X_prod[:, :3] = X_prod[:, :3] + rng.normal(0.7, 0.2, size=X_prod[:, :3].shape)

scores = [feature_drift_score(X_test[:, i], X_prod[:, i]) for i in range(5)]
print('Hellinger (5 primeras features):', np.round(scores, 3))
print('Alerta si alguna > 0.25')


Hellinger (5 primeras features): [0.071 0.075 0.084 0.    0.   ]
Alerta si alguna > 0.25
